In [1]:
import pandas as pd;
import numpy as np;

In [2]:
from pathlib import Path
BASE_DIR = Path.cwd()

In [3]:
banking = pd.read_excel(
    BASE_DIR / "Bank_Fraud_Cleaned_Dataset.xlsx")
print("Dataset Loaded Successfully")

Dataset Loaded Successfully


In [4]:
# Missing Value Check

print("\n--- Missing Value Check ---")

missing_values = banking.isnull().sum()

print(missing_values[missing_values > 0])


--- Missing Value Check ---
Series([], dtype: int64)


In [5]:
# Duplicate Value Check

print("\n--- Duplicate Check ---")

duplicate_count = banking.duplicated().sum()

print("Duplicate rows:", duplicate_count)


--- Duplicate Check ---
Duplicate rows: 0


In [6]:
# Invalid Value Check

print("\n--- Invalid Value Checks ---")

print("Invalid Age:", ((banking["Age"] < 18) | (banking["Age"] > 100)).sum())

print("Invalid Transaction Amount:",
      (banking["Transaction_Amount"] < 0).sum())

print("Invalid Account Balance:",
      (banking["Account_Balance"] < 0).sum())

print("Invalid Is_Fraud values:",
      (~banking["Is_Fraud"].isin([0, 1])).sum())

print("Invalid Transaction Currency:",
      banking["Transaction_Currency"].isnull().sum())


--- Invalid Value Checks ---
Invalid Age: 0
Invalid Transaction Amount: 0
Invalid Account Balance: 0
Invalid Is_Fraud values: 0
Invalid Transaction Currency: 0


In [7]:
# Datatype Check

print("\n--- Datatype Check ---")

print(banking.dtypes)


--- Datatype Check ---
Customer_ID                           str
Customer_Name                         str
Gender                                str
Age                                 int64
State                                 str
City                                  str
Bank_Branch                           str
Account_Type                          str
Transaction_ID                        str
Transaction_Date           datetime64[us]
Transaction_Time                      str
Transaction_Amount                float64
Merchant_ID                           str
Transaction_Type                      str
Merchant_Category                     str
Account_Balance                   float64
Transaction_Device                    str
Transaction_Location                  str
Device_Type                           str
Is_Fraud                            int64
Transaction_Currency                  str
Customer_Contact                      str
Transaction_Description               str
Customer_E

In [8]:
# Transaction_ID Uniqueness Check

print("\n--- Transaction_ID Uniqueness Check ---")

total_transactions = banking["Transaction_ID"].count()
unique_transaction_ids = banking["Transaction_ID"].nunique()
duplicate_transaction_ids = total_transactions - unique_transaction_ids

print("Total Transaction IDs:", total_transactions)
print("Unique Transaction IDs:", unique_transaction_ids)
print("Duplicate Transaction IDs:", duplicate_transaction_ids)


--- Transaction_ID Uniqueness Check ---
Total Transaction IDs: 200000
Unique Transaction IDs: 199999
Duplicate Transaction IDs: 1


In [9]:
# Find duplicated Transaction_ID values

duplicate_transaction_ids = banking[
    banking["Transaction_ID"].duplicated(keep=False)
].sort_values("Transaction_ID")

print("\n--- Duplicate Transaction_ID Details ---")
print(
    duplicate_transaction_ids[
        ["Transaction_ID", "Customer_ID", "Transaction_Amount", "Transaction_Date", "Is_Fraud"]
    ]
)


--- Duplicate Transaction_ID Details ---
                              Transaction_ID  \
199808  b16e585d-31a8-4324-96e5-32f4c3d22cde   
199809  b16e585d-31a8-4324-96e5-32f4c3d22cde   

                                 Customer_ID  Transaction_Amount  \
199808  0a536c93-98f5-4bc7-8662-9f32b8c5df54            22429.63   
199809  0a536c93-98f5-4bc7-8662-9f32b8c5df54            22429.63   

       Transaction_Date  Is_Fraud  
199808       2025-01-07         0  
199809       2025-01-07         0  


In [10]:
print("\n--- Comparing Duplicate Transaction_ID Rows ---")

print(
    duplicate_transaction_ids[
        duplicate_transaction_ids["Transaction_ID"] ==
        "b16e585d-31a8-4324-96e5-32f4c3d22cde"
    ].T
)


--- Comparing Duplicate Transaction_ID Rows ---
                                                        199808  \
Customer_ID               0a536c93-98f5-4bc7-8662-9f32b8c5df54   
Customer_Name                                    Nakul Dhingra   
Gender                                                  Female   
Age                                                         70   
State                                                  Mizoram   
City                                                    Aizawl   
Bank_Branch                                      Aizawl Branch   
Account_Type                                          Business   
Transaction_ID            b16e585d-31a8-4324-96e5-32f4c3d22cde   
Transaction_Date                           2025-01-07 00:00:00   
Transaction_Time                                      16:16:16   
Transaction_Amount                                    22429.63   
Merchant_ID              f6ce9f93-e3f9-4f3f-bf47-36e8efdafbe2c   
Transaction_Type           

In [11]:
# Check which columns differ between duplicate Transaction_ID rows

duplicate_rows = banking[
    banking["Transaction_ID"] ==
    "b16e585d-31a8-4324-96e5-32f4c3d22cde"
].copy()

# Compare the two rows
different_columns = []

for column in banking.columns:
    if duplicate_rows[column].iloc[0] != duplicate_rows[column].iloc[1]:
        different_columns.append(column)

print("\n--- Columns with Different Values ---")
print(different_columns)


--- Columns with Different Values ---
['Transaction_Type', 'Customer_Contact']


In [12]:
print("\n--- Different Values for Duplicate Transaction_ID ---")

print(
    duplicate_rows[
        ["Transaction_ID", "Transaction_Type", "Customer_Contact"]
    ].to_string(index=False)
)


--- Different Values for Duplicate Transaction_ID ---
                      Transaction_ID Transaction_Type Customer_Contact
b16e585d-31a8-4324-96e5-32f4c3d22cde         Transfer   +9194386XXXXXX
b16e585d-31a8-4324-96e5-32f4c3d22cde     Bill Payment   +9196407XXXXXX


In [13]:
# 1. Hour

banking["Transaction_Time"] = pd.to_datetime(
    banking["Transaction_Time"],
    format="%H:%M:%S"
)

banking["Hour"] = banking["Transaction_Time"].apply(
    lambda x: x.hour
)

print(banking[["Transaction_Time", "Hour"]].head())

     Transaction_Time  Hour
0 1900-01-01 16:04:07    16
1 1900-01-01 17:14:53    17
2 1900-01-01 03:09:52     3
3 1900-01-01 12:27:02    12
4 1900-01-01 18:30:46    18


In [14]:
2. # Age_Group

conditions = [
    banking["Age"].between(18, 25),
    banking["Age"].between(26, 40),
    banking["Age"].between(41, 55),
    banking["Age"].between(56, 70)
]

choices = [
    "Young",
    "Adult",
    "Middle Age",
    "Senior"
]

banking["Age_Group"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

print(banking[["Age", "Age_Group"]].head(10))

   Age   Age_Group
0   60      Senior
1   51  Middle Age
2   20       Young
3   57      Senior
4   43  Middle Age
5   54  Middle Age
6   61      Senior
7   32       Adult
8   52  Middle Age
9   32       Adult


In [15]:
# 3. Day_Name

banking["Day_Name"] = banking["Transaction_Date"].dt.day_name()

print(
    banking[["Transaction_Date", "Day_Name"]].head(10)
)

  Transaction_Date  Day_Name
0       2025-01-23  Thursday
1       2025-01-11  Saturday
2       2025-01-25  Saturday
3       2025-01-19    Sunday
4       2025-01-30  Thursday
5       2025-01-25  Saturday
6       2025-01-04  Saturday
7       2025-01-16  Thursday
8       2025-01-25  Saturday
9       2025-01-02  Thursday


In [16]:
# 4.Month_Name

banking["month_name"] = banking["Transaction_Date"].dt.month_name()

print(
    banking[["Transaction_Date", "month_name"]].head(10)
)

  Transaction_Date month_name
0       2025-01-23    January
1       2025-01-11    January
2       2025-01-25    January
3       2025-01-19    January
4       2025-01-30    January
5       2025-01-25    January
6       2025-01-04    January
7       2025-01-16    January
8       2025-01-25    January
9       2025-01-02    January


In [17]:
# 5. Transaction_Size

conditions = [
    banking["Transaction_Amount"] < 25000,
    (banking["Transaction_Amount"] >= 25000) &
    (banking["Transaction_Amount"] < 50000),
    (banking["Transaction_Amount"] >= 50000) &
    (banking["Transaction_Amount"] < 75000),
    banking["Transaction_Amount"] >= 75000
]

choices = [
    "Low",
    "Medium",
    "High",
    "Very High"
]

banking["Transaction_Size"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

print(
    banking[
        ["Transaction_Amount", "Transaction_Size"]
    ].head(10)
)

print("\nTransaction Size Categories:")
print(banking["Transaction_Size"].value_counts())

   Transaction_Amount Transaction_Size
0            32415.45           Medium
1            43622.60           Medium
2            63062.56             High
3            14000.72              Low
4            18335.16              Low
5             9711.15              Low
6            94677.01        Very High
7            67704.28             High
8            72953.45             High
9             5689.02              Low

Transaction Size Categories:
Transaction_Size
Medium       50685
High         50292
Very High    49528
Low          49495
Name: count, dtype: int64


In [18]:
# 6. Balance

conditions = [
    banking["Account_Balance"] < 25000,
    (banking["Account_Balance"] >= 25000) &
    (banking["Account_Balance"] < 50000),
    (banking["Account_Balance"] >= 50000) &
    (banking["Account_Balance"] < 75000),
    banking["Account_Balance"] >= 75000
]

choices = [
    "Low",
    "Medium",
    "High",
    "Very High"
]

banking["Balance"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

print(
    banking[
        ["Account_Balance", "Balance"]
    ].head(10)
)

print("\nBalance Categories:")
print(banking["Balance"].value_counts())

   Account_Balance Balance
0         74557.27    High
1         74622.66    High
2         66817.99    High
3         58177.08    High
4         16108.56     Low
5         61258.85    High
6         36313.61  Medium
7         16948.73     Low
8         18138.71     Low
9         65801.35    High

Balance Categories:
Balance
Medium       52887
High         52614
Very High    52400
Low          42099
Name: count, dtype: int64


In [19]:
print("Unknown Transaction_Size:",
      (banking["Transaction_Size"] == "Unknown").sum())

print("Unknown Balance:",
      (banking["Balance"] == "Unknown").sum())

Unknown Transaction_Size: 0


Unknown Balance: 0


In [20]:
# 7. Time_of_Day

conditions = [
    banking["Hour"].between(0, 5),
    banking["Hour"].between(6, 11),
    banking["Hour"].between(12, 17),
    banking["Hour"].between(18, 23)
]

choices = [
    "Late Night",
    "Morning",
    "Afternoon",
    "Evening"
]

banking["Time_of_Day"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

print(
    banking[
        ["Hour", "Time_of_Day"]
    ].head(10)
)

   Hour Time_of_Day
0    16   Afternoon
1    17   Afternoon
2     3  Late Night
3    12   Afternoon
4    18     Evening
5     6     Morning
6     0  Late Night
7     4  Late Night
8    10     Morning
9     4  Late Night


In [21]:
# 8. Balance Utilization

banking["Balance_Utilization"] = (
    banking["Transaction_Amount"] /
    banking["Account_Balance"]
)

print(
    banking[
        ["Transaction_Amount",
         "Account_Balance",
         "Balance_Utilization"]
    ].head(10)
)

# Balance Utilization Validation

print("\n--- Balance Utilization Validation ---")

print("Minimum:", banking["Balance_Utilization"].min())
print("Maximum:", banking["Balance_Utilization"].max())
print("Missing values:", banking["Balance_Utilization"].isnull().sum())
print("Infinite values:", np.isinf(banking["Balance_Utilization"]).sum())
print("Negative values:", (banking["Balance_Utilization"] < 0).sum())

   Transaction_Amount  Account_Balance  Balance_Utilization
0            32415.45         74557.27             0.434772
1            43622.60         74622.66             0.584576
2            63062.56         66817.99             0.943796
3            14000.72         58177.08             0.240657
4            18335.16         16108.56             1.138225
5             9711.15         61258.85             0.158526
6            94677.01         36313.61             2.607205
7            67704.28         16948.73             3.994652
8            72953.45         18138.71             4.021976
9             5689.02         65801.35             0.086457

--- Balance Utilization Validation ---
Minimum: 0.00014479721714063208
Maximum: 23.901711123442617
Missing values: 0
Infinite values: 0
Negative values: 0


In [22]:
# 9. Rule-Based / Heuristic Risk Score
# The Risk Score is a manually defined heuristic scoring system.
# It is based on predefined transaction and behavioural risk rules.
# It is NOT generated by a machine learning prediction model.
# Risk Score Weight Justification
# The weights are analyst-defined heuristic values based on the relative severity of each risk indicator.
# Stronger risk indicators such as very high transaction amounts and high balance utilization receive higher weights.
# Other indicators such as late-night transactions and high balances receive comparatively lower weights.
# These weights are not learned from historical data and are not generated by a machine learning model.

banking["Risk_Score"] = 0

banking.loc[
    banking["Transaction_Size"] == "Very High",
    "Risk_Score"
] += 30

banking.loc[
    banking["Transaction_Size"] == "High",
    "Risk_Score"
] += 20

banking.loc[
    banking["Time_of_Day"] == "Late Night",
    "Risk_Score"
] += 20

banking.loc[
    banking["Balance"] == "Very High",
    "Risk_Score"
] += 15

banking.loc[
    banking["Balance_Utilization"] > 2,
    "Risk_Score"
] += 35

In [23]:
# 10. Risk Level

conditions = [
    banking["Risk_Score"] <= 20,
    banking["Risk_Score"].between(21, 50),
    banking["Risk_Score"] > 50
]

choices = [
    "Low Risk",
    "Medium Risk",
    "High Risk"
]

banking["Risk_Level"] = np.select(
    conditions,
    choices,
    default="Unknown"
)

print(
    banking[
        ["Risk_Score", "Risk_Level"]
    ].head(10)
)

print(banking["Risk_Level"].value_counts())

   Risk_Score   Risk_Level
0           0     Low Risk
1           0     Low Risk
2          40  Medium Risk
3           0     Low Risk
4           0     Low Risk
5           0     Low Risk
6          85    High Risk
7          75    High Risk
8          55    High Risk
9          20     Low Risk
Risk_Level
Low Risk       101589
Medium Risk     54676
High Risk       43735
Name: count, dtype: int64


In [24]:
# Validating Risk Score using fraud rates by Risk Level

risk_summary = banking.groupby("Risk_Level").agg(
    Total_Transactions=("Transaction_ID", "count"),
    Fraud_Transactions=("Is_Fraud", "sum")
).reset_index()

risk_summary["Fraud_Rate"] = (
    risk_summary["Fraud_Transactions"] /
    risk_summary["Total_Transactions"]
) * 100

print(risk_summary)

    Risk_Level  Total_Transactions  Fraud_Transactions  Fraud_Rate
0    High Risk               43735                3506    8.016463
1     Low Risk              101589                3435    3.381272
2  Medium Risk               54676                3147    5.755725


In [25]:
# Removing unnecessary personal information from analytical layer

pii_columns = [
    "Customer_Name",
    "Customer_Contact",
    "Customer_Email"
]

banking = banking.drop(columns=pii_columns)

print("\n--- Personal Information Removed ---")
print("Removed columns:", pii_columns)
print("Remaining columns:")
print(banking.columns.tolist())


--- Personal Information Removed ---
Removed columns: ['Customer_Name', 'Customer_Contact', 'Customer_Email']
Remaining columns:
['Customer_ID', 'Gender', 'Age', 'State', 'City', 'Bank_Branch', 'Account_Type', 'Transaction_ID', 'Transaction_Date', 'Transaction_Time', 'Transaction_Amount', 'Merchant_ID', 'Transaction_Type', 'Merchant_Category', 'Account_Balance', 'Transaction_Device', 'Transaction_Location', 'Device_Type', 'Is_Fraud', 'Transaction_Currency', 'Transaction_Description', 'Hour', 'Age_Group', 'Day_Name', 'month_name', 'Transaction_Size', 'Balance', 'Time_of_Day', 'Balance_Utilization', 'Risk_Score', 'Risk_Level']


In [26]:
feature_columns = [
    "Hour",
    "Age_Group",
    "Day_Name",
    "month_name",
    "Transaction_Size",
    "Balance",
    "Time_of_Day",
    "Balance_Utilization",
    "Risk_Score",
    "Risk_Level"
]

print(banking[feature_columns].head())

   Hour   Age_Group  Day_Name month_name Transaction_Size Balance Time_of_Day  \
0    16      Senior  Thursday    January           Medium    High   Afternoon   
1    17  Middle Age  Saturday    January           Medium    High   Afternoon   
2     3       Young  Saturday    January             High    High  Late Night   
3    12      Senior    Sunday    January              Low    High   Afternoon   
4    18  Middle Age  Thursday    January              Low     Low     Evening   

   Balance_Utilization  Risk_Score   Risk_Level  
0             0.434772           0     Low Risk  
1             0.584576           0     Low Risk  
2             0.943796          40  Medium Risk  
3             0.240657           0     Low Risk  
4             1.138225           0     Low Risk  


In [27]:
banking.to_excel(
    BASE_DIR / "Bank_Fraud_Feature_Engineered (Final).xlsx",
    index=False
)

print("Dataset Saved Successfully")

Dataset Saved Successfully
